In [50]:
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

load_dotenv("../.env")

client = OpenAI()
tavily_client = TavilyClient()

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [45]:
from src.research_agent import search_web
from src.research_agent import search_fact_checks
from src.research_agent import fetch_url
from src.research_agent import (
    ResearchPlan,
    create_research_plan,
    execute_research_plan,
)
from src.research_agent import deduplicate_search_results
from src.research_agent import (
    select_relevant_results,
    SearchResultSelection,
    SelectedSearchResult,
)
from src.research_agent import get_selected_results
from src.research_agent import fetch_selected_results
from src.research_agent import run_research_agent

### 1. Objetivo del Research Agent

El objetivo del Research Agent es `localizar información externa potencialmente relevante` para verificar las claims generadas previamente por el Claim Analyzer.

A diferencia del benchmark realizado con AVeriTeC, donde cada claim dispone de un conjunto de documentos asociado dentro del *knowledge store*, en el sistema final las evidencias deben localizarse dinámicamente a partir de fuentes externas.

El Research Agent recibe como entrada una claim estructurada y utiliza herramientas de búsqueda para localizar páginas, documentos y verificaciones relacionadas con dicha afirmación.

Su función principal es recopilar documentos candidatos que puedan contener información útil para la verificación factual.

En esta etapa, el Research Agent no determina si la claim es verdadera o falsa ni selecciona todavía las evidencias finales. Los documentos recuperados serán procesados posteriormente por el componente RAG, encargado de dividirlos en fragmentos, representarlos mediante embeddings y recuperar los fragmentos más relevantes para la claim.

De forma general, el flujo será:

Claim
→ Research Agent
→ búsqueda de fuentes externas
→ recuperación de documentos
→ RAG
→ evidencias relevantes
→ Evidence Verifier

### 2. Definición de la entrada y salida del Research Agent

El Research Agent recibe como entrada una claim estructurada generada previamente por el Claim Analyzer.

La información disponible para cada claim incluye:

- `id`: identificador de la claim dentro de la noticia.
- `claim`: texto de la afirmación verificable.
- `entities`: entidades relevantes asociadas a la claim.
- `date_reference`: referencia temporal cuando exista.

A partir de esta información, el Research Agent debe localizar fuentes externas potencialmente relevantes para verificar la afirmación.

La salida del componente estará formada por un conjunto de documentos candidatos obtenidos mediante búsquedas web, búsquedas específicas de verificaciones previas y recuperación del contenido de las URLs seleccionadas.

En esta etapa todavía no se determina si los documentos apoyan o contradicen la claim. La función del Research Agent es únicamente recopilar información externa que posteriormente será procesada por el componente RAG.

### 3. Herramientas del Research Agent

El Research Agent utilizará distintas herramientas para localizar y recuperar información externa relacionada con cada claim.

Las herramientas son:

- `search_web()`: realiza una búsqueda general en la web a partir de la claim y devuelve resultados potencialmente relevantes, como títulos, URLs y fragmentos de texto.

- `search_fact_checks()`: realiza una búsqueda orientada específicamente a verificaciones previas publicadas por organizaciones de fact-checking.

- `fetch_url()`: recupera el contenido textual de una URL seleccionada previamente.

Estas herramientas cumplen funciones diferentes dentro del proceso de investigación.

`search_web()` y `search_fact_checks()` permiten descubrir fuentes potencialmente útiles, mientras que `fetch_url()` permite obtener el contenido completo de aquellas páginas que se consideren relevantes.

El Research Agent será posteriormente el componente encargado de decidir qué búsquedas realizar y qué herramientas utilizar para cada claim.

#### 3.1. Diseño de `search_web()`

La herramienta `search_web()` será responsable de realizar búsquedas generales en Internet a partir de una consulta generada para una claim.

Su objetivo no es determinar la veracidad de la afirmación, sino descubrir páginas potencialmente relevantes que puedan contener `información útil para su posterior verificación`.

La herramienta recibirá principalmente una consulta de búsqueda (`query`) y devolverá un conjunto limitado de resultados. Para cada resultado se conservará, siempre que esté disponible:

- `title`: título de la página.
- `url`: dirección de la fuente encontrada.
- `snippet`: fragmento de texto proporcionado por el motor de búsqueda.
- `source_type`: identificador del origen del resultado, que permitirá distinguir posteriormente entre búsquedas web generales y resultados procedentes de fact-checking.

El contenido completo de las páginas no se recupera en esta etapa. Esta responsabilidad corresponde posteriormente a `fetch_url()`.

#### 3.2. Prueba de `search_web()`

Se prueba la herramienta `search_web()` de forma aislada antes de integrarla en el Research Agent.

El objetivo es comprobar que, a partir de una consulta, la herramienta devuelve resultados web estructurados con título, URL, fragmento de texto y tipo de fuente.

In [5]:
query = "European Union cash payments ban 2027"

web_results = search_web(
    query=query,
    tavily_client=tavily_client,
    max_results=5,
)

web_results

[{'title': 'NewsPoint - EU to Restrict Large Cash Payments from 2027,...',
  'url': 'https://www.facebook.com/newspointapp/posts/eu-to-restrict-large-cash-payments-from-2027-tightening-anti-money-laundering-ru/1320867586860601',
  'snippet': "## NewsPoint's post\n\n### NewsPoint\n\n29 May  ·\n\nEU to Restrict Large Cash Payments from 2027, Tightening Anti-Money Laundering Rules!\n\nFrom 2027, cash payments of €10,000 or more will be banned across all European Union member states as part of a sweeping effort to strengthen financial transparency and combat money laundering. [...] The new rules come under the EU’s Anti-Money Laundering Regulation (AMLR), which aims to ensure that large financial transactions are fully traceable and cannot be used to conceal illicit activity.\n\nKey Changes Under the New Rules\n\nUnder the updated framework:\n\n- Cash payments of €10,000 and above will be prohibited across the EU\n\n- Transactions between €3,000 and €10,000 will require identity verificati

##### 3.2.1. Resultado de la primera búsqueda

La primera ejecución de `search_web()` devuelve correctamente una lista de resultados estructurados con los campos `title`, `url`, `snippet` y `source_type`.

Los resultados obtenidos están relacionados con la consulta realizada y contienen información sobre las restricciones a los pagos en efectivo en la Unión Europea previstas para 2027.

Sin embargo, también se observa que la búsqueda puede recuperar fuentes de `naturaleza y calidad` muy diferentes, incluyendo redes sociales, medios de comunicación, publicaciones académicas y páginas informativas.

Este comportamiento es esperado, ya que la responsabilidad de `search_web()` es descubrir documentos candidatos y no evaluar todavía la fiabilidad de las fuentes.

Los fragmentos devueltos por el buscador proporcionan información preliminar sobre el contenido de cada página, mientras que la recuperación del contenido completo se realizará posteriormente mediante `fetch_url()`.

### 3.3. Diseño de `search_fact_checks()`

La herramienta `search_fact_checks()` tiene como objetivo localizar verificaciones previas relacionadas con una claim.

A diferencia de `search_web()`, que realiza una búsqueda general en Internet, esta herramienta restringe la búsqueda a dominios especializados en fact-checking.

El objetivo es identificar artículos de **verificación** que puedan aportar contexto, fuentes o evidencias relevantes para la claim analizada.

La herramienta no determina automáticamente que una verificación previa sea correcta. Los resultados obtenidos se consideran documentos candidatos que deberán ser procesados posteriormente junto con el resto de fuentes.

#### 3.4. Prueba de `search_fact_checks()`

Se prueba la herramienta `search_fact_checks()` de forma aislada utilizando la misma consulta empleada previamente en `search_web()`.

El objetivo es comprobar que la búsqueda queda restringida a dominios especializados en fact-checking y comparar la naturaleza de los resultados obtenidos con una búsqueda web general.

In [4]:
query = "European Union cash payments ban 2027"

fact_check_results = search_fact_checks(
    query=query,
    tavily_client=tavily_client,
    max_results=5,
)

fact_check_results

[{'title': 'Medium',
  'url': 'https://shanakaanslemperera.medium.com/europes-2027-financial-reset-the-end-of-monetary-privacy-or-the-beginning-of-economic-security-2746deee3086',
  'snippet': 'Cash Transaction Prohibition: Effective 2027, Regulation 2024/1624 prohibits cash payments exceeding €10,000 throughout the European Union. This represents a significant tightening from the patchwork of national rules, some of which permitted transactions up to €15,000. The regulation applies universally\u200a—\u200ato merchants, professionals, and private individuals alike. A French citizen purchasing a used vehicle for €12,000 in cash commits a regulatory violation. An Italian jeweler accepting €11,000 [...] The European Union is executing the most comprehensive reconstruction of its monetary system since the euro’s introduction in 1999. By 2027, a regulatory trinity\u200a—\u200acash transaction caps, mandatory cryptocurrency surveillance, and a digital euro prototype\u200a—\u200awill fundamen

In [5]:
query = "European Union cash payments ban 2027 fact check"

fact_check_results = search_fact_checks(
    query=query,
    tavily_client=tavily_client,
    max_results=5,
)

fact_check_results

[{'title': 'Fact Check: Britain has not announced a ban on cash payments over 10,000 pounds | Reuters',
  'url': 'https://www.reuters.com/fact-check/britain-has-not-announced-ban-cash-payments-over-10000-pounds-2025-11-28',
  'snippet': "### Fact Check: Old train crash protest video captioned as new protest in Greece\n\nA video of a demonstration last year in Thessaloniki, Greece, on the second anniversary of \u200bthe country's deadliest train accident was misrepresented online \u200cas footage of a massive protest in August. [...] ## Browse World\n\n## Browse Business\n\n## Browse Markets\n\n## Browse Sustainability\n\n## Legal\n\n## Commentary\n\n## Technology\n\n## Investigations\n\n## Sports\n\n## Science\n\n## Lifestyle\n\n## City Memo\n\n## Graphics\n\n## Pictures\n\n## Wider Image\n\n## Podcasts\n\n## Live\n\n## Fact Check\n\n## Video\n\n## Media Center\n\n## Sponsored Content\n\n# Fact Check: Britain has not announced a ban on cash payments over 10,000 pounds\n\nReuters Fact C

##### 3.4.1. Resultado de la búsqueda de fact-checks

La herramienta `search_fact_checks()` devuelve resultados relacionados con la consulta y permite recuperar verificaciones previas cuando estas existen.

Sin embargo, las pruebas muestran que la restricción utilizada en la búsqueda no garantiza que todos los resultados recuperados correspondan realmente a artículos de fact-checking.

En algunos casos se obtienen páginas informativas, artículos generales o contenidos de otros tipos junto con verificaciones especializadas.

Por tanto, la búsqueda orientada a fact-checking permite aumentar la probabilidad de recuperar verificaciones previas, pero no debe interpretarse como una clasificación fiable del tipo de fuente.

Este comportamiento deberá tenerse en cuenta posteriormente en el Research Agent, que podrá analizar y seleccionar los resultados más relevantes antes de incorporarlos al conjunto de documentos candidatos.

#### 3.5. Diseño de `fetch_url()`

La herramienta `fetch_url()` tiene como objetivo recuperar el contenido textual de una página web previamente localizada mediante las herramientas de búsqueda.

A diferencia de `search_web()` y `search_fact_checks()`, esta herramienta no descubre nuevas fuentes. Su función es acceder a una URL concreta y obtener el contenido necesario para que pueda ser procesado posteriormente por el componente RAG.

El resultado debe conservar, siempre que sea posible:

- `url`: dirección de la página recuperada.
- `title`: título del documento.
- `text`: contenido textual de la página.
- `source_type`: tipo de origen asociado al documento.

Los documentos recuperados mediante esta herramienta constituirán la entrada del pipeline de recuperación de evidencias.

Antes de probar fetch_url(), vamos a hacer una prueba porque necesitamos saber que devuelve exactamente tavily_client.extract(), que es la versión de Tavily para extract, que es lo que usaremos dentro de la función fetch_url()

In [6]:
test_url = web_results[0]["url"]

extract_response = tavily_client.extract(
    urls=[test_url]
)

extract_response

{'results': [{'url': 'https://www.facebook.com/newspointapp/posts/eu-to-restrict-large-cash-payments-from-2027-tightening-anti-money-laundering-ru/1320867586860601',
   'title': 'EU to Restrict Large Cash Payments from 2027, Tightening ...',
   'raw_content': "## NewsPoint's post\n\n---\n\n### [**NewsPoint**](https://www.facebook.com/newspointapp?__cft__[0]=AZhBIjCh2h1Zz0DYZ7lklrip1i4PC4REN01icycFT8_ykMB74puLTGDc275kMfqQNkPqY-4J2hdZ7HGhVaJ82J8QxX4AdsBGn1Zd4N9465WkwGRHOmUyOH-k7d10u83xUl7CEIZcn_54Mw50vZw0vjyKOIf5ZnRt&__tn__=-UC%2CP-R)\n\n[29 May](https://www.facebook.com/newspointapp/posts/pfbid02ZW9Q54WGGgv8di5cVEB8xeHYv33XJKogbaRnUHtKDA24W99C6jE1o2ByzyM19iFsl?__cft__[0]=AZhBIjCh2h1Zz0DYZ7lklrip1i4PC4REN01icycFT8_ykMB74puLTGDc275kMfqQNkPqY-4J2hdZ7HGhVaJ82J8QxX4AdsBGn1Zd4N9465WkwGRHOmUyOH-k7d10u83xUl7CEIZcn_54Mw50vZw0vjyKOIf5ZnRt&__tn__=%2CO%2CP-R)\xa0 ·\n\nEU to Restrict Large Cash Payments from 2027, Tightening Anti-Money Laundering Rules!\n\nFrom 2027, cash payments of €10,000 or more

##### 3.5.1. Prueba de `fetch_url()`

Una vez comprobada la estructura devuelta por `tavily_client.extract()`, se prueba la función `fetch_url()` utilizando una URL obtenida previamente mediante `search_web()`.

El objetivo es verificar que la función recupera correctamente el contenido de la página y lo transforma en una estructura simplificada que pueda utilizarse posteriormente como entrada del componente RAG.

In [10]:
test_url = web_results[0]["url"]

document = fetch_url(
    url=test_url,
    tavily_client=tavily_client,
    source_type="web",
)

document

{'url': 'https://www.facebook.com/newspointapp/posts/eu-to-restrict-large-cash-payments-from-2027-tightening-anti-money-laundering-ru/1320867586860601',
 'title': 'EU to Restrict Large Cash Payments from 2027, Tightening ...',
 'text': "## NewsPoint's post\n\n---\n\n### [**NewsPoint**](https://www.facebook.com/newspointapp?__cft__[0]=AZhBIjCh2h1Zz0DYZ7lklrip1i4PC4REN01icycFT8_ykMB74puLTGDc275kMfqQNkPqY-4J2hdZ7HGhVaJ82J8QxX4AdsBGn1Zd4N9465WkwGRHOmUyOH-k7d10u83xUl7CEIZcn_54Mw50vZw0vjyKOIf5ZnRt&__tn__=-UC%2CP-R)\n\n[29 May](https://www.facebook.com/newspointapp/posts/pfbid02ZW9Q54WGGgv8di5cVEB8xeHYv33XJKogbaRnUHtKDA24W99C6jE1o2ByzyM19iFsl?__cft__[0]=AZhBIjCh2h1Zz0DYZ7lklrip1i4PC4REN01icycFT8_ykMB74puLTGDc275kMfqQNkPqY-4J2hdZ7HGhVaJ82J8QxX4AdsBGn1Zd4N9465WkwGRHOmUyOH-k7d10u83xUl7CEIZcn_54Mw50vZw0vjyKOIf5ZnRt&__tn__=%2CO%2CP-R)\xa0 ·\n\nEU to Restrict Large Cash Payments from 2027, Tightening Anti-Money Laundering Rules!\n\nFrom 2027, cash payments of €10,000 or more will be banned across a

##### 3.5.2. Resultado de la prueba de `fetch_url()`

La función `fetch_url()` recupera correctamente el contenido asociado a una URL previamente localizada mediante las herramientas de búsqueda.

La salida obtenida contiene los campos `url`, `title`, `text` y `source_type`, proporcionando una representación simplificada del documento que puede ser utilizada posteriormente por el **componente RAG**.

A diferencia de los snippets obtenidos durante la búsqueda, el campo `text` contiene el contenido textual completo recuperado de la página.

Con esta prueba quedan validadas las tres herramientas básicas de investigación: `search_web()`, `search_fact_checks()` y `fetch_url()`.

### 4. Diseño de la lógica del Research Agent

Una vez validadas individualmente las herramientas de búsqueda y recuperación de contenido, el siguiente paso consiste en construir la lógica del Research Agent.

El Research Agent recibe una claim generada previamente por el Claim Analyzer y utiliza un LLM para decidir qué información debe buscar para investigarla.

Su responsabilidad será:

1. Analizar la claim y su contexto.
2. Generar consultas de búsqueda adecuadas.
3. Decidir cuándo realizar una búsqueda web general y cuándo buscar verificaciones previas.
4. Utilizar `search_web()` y `search_fact_checks()` para localizar fuentes potencialmente relevantes.
5. Seleccionar URLs candidatas.
6. Utilizar `fetch_url()` para recuperar el contenido de las páginas seleccionadas.
7. Devolver el conjunto de documentos candidatos que será procesado posteriormente por el componente RAG.

El Research Agent no determina si la claim es verdadera o falsa. Su responsabilidad termina con la recopilación de documentos potencialmente útiles para la verificación.

#### 4.1. Prueba de `create_research_plan()`

Se prueba de forma aislada la generación del plan de investigación a partir de una claim.

El objetivo es comprobar que el Research Agent genera consultas diferenciadas para búsqueda web general y para búsqueda de verificaciones previas, conservando las entidades y referencias temporales relevantes.

Hasta este punto, donde todavía no tenemos el agente `Research_Agent` completo, estamos probando la parte inicial, es decir, esta parte del agente que estamos probando ahora en este punto responde a: `¿Qué debería buscar para investigar esta claim?`. En este punto, todavía no busca nada.

In [14]:
claim = "La Unión Europea prohibirá completamente los pagos en efectivo a partir de 2027."

entities = ["Unión Europea"]

date_reference = "2027"

research_plan = create_research_plan(
    claim=claim,
    entities=entities,
    date_reference=date_reference,
    client=client,
)

research_plan

ResearchPlan(web_queries=['Unión Europea prohibición pagos en efectivo 2027 reglamento límite efectivo', 'EU cash payments ban 2027 regulation cash payment limit', 'Consejo de la UE paquete antiblanqueo pagos en efectivo límite 10000 euros entrada en vigor', 'Reglamento UE prevención blanqueo capitales pagos en efectivo 2027 texto oficial'], fact_check_queries=['fact check Unión Europea prohibirá pagos en efectivo 2027', 'verificación UE prohibición efectivo 2027', 'EU cash ban 2027 fact check', 'AFP Factuel pagos efectivo Unión Europea 2027'])

#### 4.1.1. Resultado del primer plan de investigación

El Research Agent genera correctamente dos grupos diferenciados de consultas: búsquedas web generales y búsquedas orientadas a verificaciones previas.

Las consultas web cubren diferentes perspectivas de la claim, incluyendo búsquedas generales, referencias a normativa europea y fuentes oficiales.

Las consultas destinadas a fact-checking también están correctamente orientadas a localizar verificaciones relacionadas con la afirmación.

No obstante, se observan dos aspectos a mejorar:

- el número de consultas generadas puede resultar excesivo si el proceso se aplica a múltiples claims;
- algunas consultas introducen directamente nombres de organizaciones de fact-checking, aunque estas no formen parte de la claim original.

Por este motivo, se limitará el número de consultas generadas y se priorizarán queries temáticas, dejando la selección de dominios especializados a la herramienta `search_fact_checks()`.

In [17]:
claim = "La Unión Europea prohibirá completamente los pagos en efectivo a partir de 2027."

entities = ["Unión Europea"]

date_reference = "2027"

research_plan_2 = create_research_plan(
    claim=claim,
    entities=entities,
    date_reference=date_reference,
    client=client,
)

research_plan_2

ResearchPlan(web_queries=['Unión Europea pagos en efectivo prohibición 2027 reglamento límite efectivo', 'EU cash payments ban 2027 anti-money laundering regulation cash payment limit', 'Reglamento Unión Europea 2027 pagos en efectivo límite 10000 euros'], fact_check_queries=['verificación afirmación Unión Europea prohibirá completamente pagos en efectivo a partir de 2027', 'fact check EU completely ban cash payments from 2027'])

#### 4.1.2. Ajuste del plan de investigación

Tras la primera prueba se modificaron las instrucciones del Research Agent para limitar el número de consultas generadas y evitar la inclusión de organizaciones concretas de fact-checking cuando no fueran necesarias. En las instrucciones se añadieron estos puntos:

* Generate at most 3 web queries and at most 2 fact-check queries.
* Fact-check queries should describe the claim to be verified.
* Do not include the name of a specific fact-checking organization unless it is explicitly relevant to the claim.

La nueva ejecución genera tres consultas web y dos consultas orientadas a verificación previa.

Las consultas obtenidas mantienen las entidades y el contexto temporal de la claim, cubren diferentes formulaciones relevantes y presentan una menor redundancia que en la primera versión.

Además, las consultas de fact-checking se centran ahora en describir la afirmación que debe verificarse, dejando la selección de dominios especializados a la herramienta `search_fact_checks()`.

Por tanto, esta versión del plan de investigación se considera adecuada para continuar con la ejecución automática de las herramientas de búsqueda.

### 4.2. Ejecución del plan de investigación

Una vez generado el `ResearchPlan`, el siguiente paso consiste en ejecutar las consultas propuestas por el Research Agent.

Cada elemento de `web_queries` se ejecutará mediante `search_web()`, mientras que las consultas incluidas en `fact_check_queries` se ejecutarán mediante `search_fact_checks()`.

En esta fase se conservan los resultados obtenidos sin recuperar todavía el contenido completo de las páginas. Esto permite analizar previamente la calidad, relevancia y posible redundancia de las fuentes encontradas antes de utilizar `fetch_url()`.

### 4.3. Prueba de `execute_research_plan()`

Se ejecuta automáticamente el plan de investigación generado previamente por el Research Agent.

Las consultas incluidas en `web_queries` se envían a `search_web()`, mientras que las consultas incluidas en `fact_check_queries` se procesan mediante `search_fact_checks()`.

En esta fase todavía no se recupera el contenido completo de las URLs. El objetivo es analizar primero el conjunto de resultados obtenidos, comprobar su relevancia y observar posibles duplicados entre consultas.

In [22]:
research_results = execute_research_plan(
    research_plan=research_plan_2,
    tavily_client=tavily_client,
    max_results_per_query=5,
)

research_results

{'web_results': [{'title': 'La UE cambia las normas: los comercios en Europa, a partir de 2027, tendrán un nuevo límite de pagos en efectivo',
   'url': 'https://www.elespanol.com/sociedad/20260531/ue-cambia-normas-comercios-europa-partir-nuevo-limite-pagos-efectivo/1003744262603_0.html',
   'snippet': 'Este límite se aplicará a toda la UE a más tardar en el verano de 2027, tres años después de la entrada en vigor del reglamento. Es importante subrayar que el tope europeo es un máximo, y cada país puede establecer límites más estrictos.\n\nDe hecho, España ya prohíbe pagos en efectivo de más de 1.000 euros cuando una de las partes es un empresario o profesional, según la Ley Antifraude 11/2021. Para particulares no residentes, el límite es de 10.000 euros. [...] Miguel Villacorta\n\nPublicada\n\nLas claves\n\n### Las claves Generado con IA\n\nLa Unión Europea impondrá en 2027 un límite de 10.000 euros por operación para pagos en efectivo, buscando combatir el blanqueo de capitales y el

#### 4.3.1. Análisis de los resultados obtenidos

La ejecución automática del plan de investigación genera resultados diferenciados para las búsquedas web generales y para las búsquedas orientadas a fact-checking.

Las búsquedas web recuperan principalmente documentos relacionados con la claim, incluyendo medios de comunicación, publicaciones especializadas y fuentes institucionales o regulatorias.

Sin embargo, también se observan resultados duplicados o muy similares entre consultas.

En las búsquedas de fact-checking se recuperan algunas verificaciones potencialmente relevantes, pero también numerosos resultados que pertenecen a dominios de fact-checking y no guardan una relación directa con la claim.

Por tanto, antes de recuperar el contenido completo de las URLs mediante `fetch_url()`, es necesario incorporar una `etapa de deduplicación y selección de resultados relevantes`.

Esta etapa permitirá reducir el número de documentos procesados posteriormente y evitar que información claramente irrelevante llegue al componente RAG.

### 4.4. Deduplicación de resultados

Antes de seleccionar las fuentes relevantes, se eliminan URLs repetidas entre los resultados obtenidos para una misma categoría de búsqueda.

La deduplicación se realiza de forma determinista utilizando la URL como identificador, conservando únicamente la primera aparición de cada recurso.

In [25]:
unique_web_results = deduplicate_search_results(
    research_results["web_results"]
)

unique_fact_check_results = deduplicate_search_results(
    research_results["fact_check_results"]
)

In [27]:
print("Web Results:",len(research_results["web_results"]),"->", len(unique_web_results))

Web Results: 15 -> 14


In [28]:
print("Fact Checking results",len(research_results["fact_check_results"]),"->", len(unique_fact_check_results))

Fact Checking results 10 -> 10


#### 4.4.1. Resultado de la deduplicación

La deduplicación reduce los resultados web de 15 a 14 documentos únicos, mientras que los resultados de fact-checking se mantienen en 10.

Esto indica que la redundancia exacta por URL es limitada en esta prueba.

Sin embargo, el análisis anterior mostró que varios resultados, especialmente dentro de las búsquedas de fact-checking, no guardan una relación suficiente con la claim. Por tanto, la principal necesidad no es eliminar duplicados, sino seleccionar los resultados realmente relevantes antes de recuperar su contenido completo.

### 4.5. Selección de resultados relevantes

Tras eliminar los duplicados, se utiliza un LLM para seleccionar únicamente los resultados que pueden aportar información útil para verificar la claim.

La selección se realiza a partir del título, snippet, URL y tipo de fuente de cada candidato. El modelo devuelve los identificadores de los resultados seleccionados y una breve justificación para cada uno.

En esta fase todavía no se recupera el contenido completo de las páginas.

In [33]:
claim

'La Unión Europea prohibirá completamente los pagos en efectivo a partir de 2027.'

In [31]:
selection = select_relevant_results(
    claim=claim,
    web_results=unique_web_results,
    fact_check_results=unique_fact_check_results,
    client=client,
)

selection

SearchResultSelection(selected_results=[SelectedSearchResult(result_id=7, reason='Verificación directamente centrada en el bulo de que Bruselas habría prohibido el efectivo; aclara que la medida es un límite de 10.000 euros para pagos en efectivo a empresas desde 2027, no una prohibición total.'), SelectedSearchResult(result_id=10, reason='Cita el Reglamento (UE) 2024/1624, la fecha de aplicación (10 de julio de 2027), el umbral de 10.000 euros y precisa que afecta principalmente a transacciones comerciales, con excepciones relevantes.'), SelectedSearchResult(result_id=9, reason='Fuente académica que identifica el artículo 80 del Reglamento y describe el límite de pagos en efectivo de hasta 10.000 euros desde el 10 de julio de 2027; sirve para contrastar el alcance jurídico de la afirmación.'), SelectedSearchResult(result_id=6, reason='Análisis jurídico del reglamento que explica que se prohíben pagos superiores a 10.000 euros, no todo pago en efectivo, y que los Estados pueden imponer

In [32]:
all_candidates = unique_web_results + unique_fact_check_results

for selected in selection.selected_results:
    result = all_candidates[selected.result_id]

    print("RESULT ID:", selected.result_id)
    print("TITLE:", result["title"])
    print("URL:", result["url"])
    print("SOURCE TYPE:", result["source_type"])
    print("REASON:", selected.reason)
    print("-" * 80)

RESULT ID: 7
TITLE: No, Brussels didn't just criminalise cash  | Euronews
URL: https://www.euronews.com/my-europe/2025/11/13/no-brussels-didnt-just-criminalise-cash
SOURCE TYPE: web
REASON: Verificación directamente centrada en el bulo de que Bruselas habría prohibido el efectivo; aclara que la medida es un límite de 10.000 euros para pagos en efectivo a empresas desde 2027, no una prohibición total.
--------------------------------------------------------------------------------
RESULT ID: 10
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
URL: https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_450803.html
SOURCE TYPE: web
REASON: Cita el Reglamento (UE) 2024/1624, la fecha de aplicación (10 de julio de 2027), el umbral de 10.000 euros y precisa que afecta principalmente a transacciones comerciales, con excepciones relevantes.
---

#### 4.5.1. Resultado de la selección de fuentes

El componente de selección reduce los 24 resultados deduplicados a seis candidatos considerados útiles para la verificación de la claim.

Las fuentes seleccionadas incluyen artículos periodísticos, análisis jurídicos, una publicación académica y una fuente institucional. La mayoría aborda directamente el límite europeo de 10.000 euros para determinados pagos en efectivo y su aplicación a partir de 2027.

Es relevante observar que ninguno de los resultados procedentes de la búsqueda específica de fact-checking fue seleccionado. Esto se debe a que los resultados recuperados por dicha búsqueda presentaban una baja relación con la claim analizada. El modelo priorizó, en cambio, resultados obtenidos mediante la búsqueda web general que contenían información más directamente relacionada con la afirmación.

Este comportamiento confirma que el campo `source_type` representa el origen de la búsqueda y no constituye por sí mismo una valoración de la relevancia o calidad de la fuente.

La selección se considera suficientemente adecuada para continuar con la recuperación del contenido completo de las URLs seleccionadas.

### 4.6. Recuperación de los resultados seleccionados

Una vez identificados los `result_id` relevantes, se recuperan los resultados originales asociados a esos identificadores.

Esta etapa es determinista y no utiliza el LLM. Su objetivo es mantener las URLs y metadatos originales antes de recuperar el contenido completo de las páginas.

In [36]:
selected_results = get_selected_results(
    selection=selection,
    web_results=unique_web_results,
    fact_check_results=unique_fact_check_results,
)

In [37]:
len(selected_results)

6

In [38]:
for result in selected_results:
    print("TITLE:", result["title"])
    print("URL:", result["url"])
    print("SOURCE TYPE:", result["source_type"])
    print("SELECTION REASON:", result["selection_reason"])
    print("-" * 80)

TITLE: No, Brussels didn't just criminalise cash  | Euronews
URL: https://www.euronews.com/my-europe/2025/11/13/no-brussels-didnt-just-criminalise-cash
SOURCE TYPE: web
SELECTION REASON: Verificación directamente centrada en el bulo de que Bruselas habría prohibido el efectivo; aclara que la medida es un límite de 10.000 euros para pagos en efectivo a empresas desde 2027, no una prohibición total.
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
URL: https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_450803.html
SOURCE TYPE: web
SELECTION REASON: Cita el Reglamento (UE) 2024/1624, la fecha de aplicación (10 de julio de 2027), el umbral de 10.000 euros y precisa que afecta principalmente a transacciones comerciales, con excepciones relevantes.
----------

### 4.7. Recuperación del contenido de las fuentes seleccionadas

Una vez seleccionadas las fuentes relevantes, se recupera el contenido textual completo de cada URL mediante `fetch_url()`.

Además del contenido de la página, se conservan metadatos asociados al proceso de investigación, como la consulta que originó el resultado y la razón por la que fue seleccionado.

El resultado de esta fase constituye el conjunto de documentos candidatos que posteriormente será procesado por el componente RAG.

In [41]:
candidate_documents = fetch_selected_results(
    selected_results=selected_results,
    tavily_client=tavily_client,
)

len(candidate_documents)

6

In [42]:
for document in candidate_documents:
    print("TITLE:", document["title"])
    print("URL:", document["url"])
    print("TEXT LENGTH:", len(document["text"]) if document["text"] else 0)
    print("-" * 80)

TITLE: No, Brussels didn't just criminalise cash | Euronews
URL: https://www.euronews.com/my-europe/2025/11/13/no-brussels-didnt-just-criminalise-cash
TEXT LENGTH: 21972
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
URL: https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_450803.html
TEXT LENGTH: 9921
--------------------------------------------------------------------------------
TITLE: Payment by Cash or Card? Restrictions on Cash Circulation in ...
URL: https://link.springer.com/article/10.1007/s11196-025-10374-w
TEXT LENGTH: 84313
--------------------------------------------------------------------------------
TITLE: EU Regulation On Cash Payment Limits - Money Laundering
URL: https://www.mondaq.com/money-laundering/1717828/eu-regulation-on-cash-p

#### 4.7.1. Resultado de la recuperación de documentos

Las seis fuentes seleccionadas pudieron recuperarse correctamente mediante `fetch_url()`.

El tamaño de los documentos recuperados varía considerablemente, desde aproximadamente 5.000 caracteres hasta más de 80.000 caracteres en el caso de una publicación académica.

Este volumen de información confirma la necesidad de incorporar posteriormente un componente RAG que divida los documentos en fragmentos y recupere únicamente las evidencias más relevantes para cada claim.

Los documentos obtenidos en esta etapa constituyen la salida del Research Agent y la entrada del componente de recuperación de evidencias.

### 4.8. Prueba end-to-end del Research Agent

Una vez validadas de forma individual las distintas etapas del Research Agent, se ejecuta el flujo completo mediante `run_research_agent()`.

El objetivo de esta prueba es comprobar que, a partir de una claim y sus metadatos, el componente es capaz de generar un plan de investigación, ejecutar las búsquedas, eliminar duplicados, seleccionar las fuentes relevantes y recuperar su contenido completo sin intervención manual.

La salida de esta función corresponde al conjunto de documentos candidatos que posteriormente serán procesados por el componente RAG.

In [46]:
candidate_documents_end_to_end = run_research_agent(
    claim=claim,
    entities=entities,
    date_reference=date_reference,
    client=client,
    tavily_client=tavily_client,
)

len(candidate_documents_end_to_end)

6

In [47]:
for document in candidate_documents_end_to_end:
    print("TITLE:", document["title"])
    print("URL:", document["url"])
    print(
        "TEXT LENGTH:",
        len(document["text"]) if document["text"] else 0
    )
    print("SOURCE TYPE:", document["source_type"])
    print("-" * 80)

TITLE: Prevención del abuso del sistema financiero para fines de ...
URL: https://eur-lex.europa.eu/ES/legal-content/summary/preventing-abuse-of-the-financial-system-for-money-laundering-and-terrorism-purposes-from-2027.html
TEXT LENGTH: 7895
SOURCE TYPE: web
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de ...
URL: https://www.eleconomista.es/economia/noticias/14002307/07/26/la-union-europea-limitara-los-pagos-en-efectivo-a-partir-de-2027-y-prohibira-comprar-con-dinero-en-metalico-cuando-se-supere-esta-cantidad.html
TEXT LENGTH: 3658
SOURCE TYPE: web
--------------------------------------------------------------------------------
TITLE: La Unión Europea limitará los pagos en efectivo a partir de 2027: la cantidad que no podrás pagar en metálico
URL: https://www.eldebate.com/economia/consumo/20260820/union-europea-limitara-pagos-efectivo-partir-2027-cantidad-no-podras-pagar-metalico-cns_4

#### 4.8.1. Resultado de la prueba end-to-end

La ejecución completa del Research Agent devuelve seis documentos candidatos relacionados con la claim analizada.

Entre las fuentes recuperadas se incluyen fuentes institucionales y regulatorias, análisis jurídicos, una publicación académica y medios de comunicación. Destaca la recuperación de recursos como EUR-Lex y SEPBLAC, directamente relacionados con la normativa europea sobre prevención del blanqueo de capitales y los límites aplicables a determinados pagos en efectivo.

La ejecución confirma que el Research Agent es capaz de realizar de forma automática las distintas etapas desarrolladas previamente: generación del plan de investigación, ejecución de búsquedas, eliminación de duplicados, selección de resultados relevantes y recuperación del contenido completo de las fuentes seleccionadas.

Aunque las fuentes concretas pueden variar entre ejecuciones debido al uso de componentes basados en LLM y a la naturaleza dinámica de las búsquedas web, el comportamiento general del sistema se mantiene estable.

Los documentos obtenidos constituyen la entrada del siguiente componente del sistema: el pipeline RAG encargado de fragmentar los documentos, generar sus representaciones vectoriales y recuperar las evidencias más relevantes para cada claim.